In [28]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "TerriblePassword12@"

# Connect to database via CRUD Module
db = AnimalShelter()

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
print("Number of records loaded:", len(df))
print("Columns in dataframe:", df.columns.tolist())


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
    html.Div(id='hidden-div', style={'display':'none'}),
    
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.H4("Noelia - CRN:10525"),
    html.Hr(),

    # Logo
    html.Center(
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()),
            style={'width': '250px', 'height': 'auto'}
        )
    ),

    # Filter + Clear Button
    html.Div([
        html.Label("Filter by Rescue Type:"),
        dcc.Dropdown(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain/Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster/Tracking', 'value': 'disaster'},
                {'label': 'All', 'value': 'reset'}
            ],
            value='reset',
            clearable=False,
            style={'width': '300px'}
        ),

        html.Button(
            "Clear Row Selection",
            id="clear-row",
            n_clicks=0,
            style={'margin-left': '20px', 'height': '40px'}
        )
    ], style={'margin-bottom': '20px'}),

    html.Hr(),

    # DataTable
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        page_action="native"
    ),

    html.Br(),
    html.Hr(),

    # Side by side: Pie chart on left, Map on right
    html.Div([
        html.Div(id='graph-id', style={'flex': '1', 'margin-right': '20px'}),
        html.Div(id='map-id', style={'flex': '1'})
    ], style={'display': 'flex', 'align-items': 'flex-start'})
])
        

#############################################
# Interaction Between Components / Controller
#############################################



# Update DataTable based on dropdown selection
@app.callback(
    Output('datatable-id','data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):

    # ----------------------------
    # MongoDB queries using CRUD
    # ----------------------------

    if filter_type == "water":
        query = {
            "breed": {"$in": ["Labrador Retriever Mix","Chesapeake Bay Retriever","Newfoundland"]},
            "sex_upon_outcome": {"$regex": "Intact Female"},
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == "mountain":
        query = {
            "breed": {"$in": ["German Shepherd","Alaskan Malamute","Old English Sheepdog","Siberian Husky","Rottweiler"]},
            "sex_upon_outcome": {"$regex": "Intact Male"},
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == "disaster":
        query = {
            "breed": {"$in": ["Doberman Pinscher","German Shepherd","Golden Retriever","Bloodhound","Rottweiler"]},
            "sex_upon_outcome": {"$regex": "Intact Male"},
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }

    else:
        query = {}  # reset → return all records

    # Run query through CRUD module
    df_filtered = pd.DataFrame.from_records(db.read(query))

    # Remove Mongo _id column
    df_filtered.drop(columns=['_id'], inplace=True, errors='ignore')

    return df_filtered.to_dict('records')

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    ### FIX ME ####
    # add code for chart of your choice (e.g. pie chart) #

    # Check if data exists
    if viewData is None or len(viewData) == 0:
        return html.P("No data to display")

    # Convert the viewData from datatable to a pandas DataFrame
    df_chart = pd.DataFrame.from_dict(viewData)

    # Count occurrences of each breed
    breed_counts = df_chart['breed'].value_counts()

    # ttop 15 breeds; everything else goes into 'Other' to make the pie chart readable bc that was crazy
    top_breeds = breed_counts.nlargest(15).index
    df_chart['breed_filtered'] = df_chart['breed'].where(df_chart['breed'].isin(top_breeds), 'Other')

    # Create the pie chart using Plotly Express
    fig = px.pie(
        df_chart,
        names='breed_filtered',
    )

    # set figure size and margins so it appears properly bc it was too small
    fig.update_layout(
        width=600,   # Adjust width as needed
        height=600,  # Adjust height as needed
        margin=dict(t=50, b=50, l=50, r=50)
    )

    # Return the Graph component with the properly sized figure
    return dcc.Graph(figure=fig)
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    # If nothing is selected, return empty list
    if not selected_columns:
        return []
    
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
##########################################
# Clear Row Selection Button Callback
##########################################
@app.callback(
    Output('datatable-id', 'selected_rows'),
    [Input('clear-row', 'n_clicks'),
     Input('filter-type', 'value')]
)
def update_selected_rows(clear_clicks, filter_value):
    ctx = dash.callback_context

    if not ctx.triggered:
        # Nothing triggered yet, return no change
        return []
    
    trigger_id = ctx.triggered[0]['prop_id'].split('.')[0]

    if trigger_id == 'clear-row':
        # Clear button clicked -> deselect all
        return []
    elif trigger_id == 'filter-type':
        # Filter changed -> deselect all
        return []
    
    return dash.no_update


##########################################
# Map callback
##########################################
@app.callback(
    Output('map-id', "children"),
    [
        Input('datatable-id', "derived_virtual_data"),
        Input('datatable-id', "derived_virtual_selected_rows"),
        Input('filter-type', "value")
    ]
)
def update_map(viewData, selected_rows, filter_value):

    # Base map (Austin, TX)
    base_map = dl.Map(
        style={'width': '1000px', 'height': '500px'},
        center=[30.75, -97.48],
        zoom=10,
        children=[dl.TileLayer(id="base-layer-id")]
    )

    # If no data available
    if not viewData:
        return [base_map]

    # Convert to DataFrame
    dff = pd.DataFrame.from_dict(viewData)

    # -------------------------------------------------
    # always show marker when row selecteda
    # -------------------------------------------------
    if selected_rows and len(selected_rows) > 0:
        row_idx = selected_rows[0]

        if row_idx < len(dff):
            lat = dff.iloc[row_idx]['location_lat']
            lon = dff.iloc[row_idx]['location_long']

            if pd.notna(lat) and pd.notna(lon):
                return [
                    dl.Map(
                        style={'width': '1000px', 'height': '500px'},
                        center=[lat, lon],
                        zoom=10,
                        children=[
                            dl.TileLayer(),
                            dl.Marker(
                                position=[lat, lon],
                                children=[
                                    dl.Tooltip(dff.iloc[row_idx]['breed']),
                                    dl.Popup([
                                        html.H1("Animal Name"),
                                        html.P(dff.iloc[row_idx]['name'])
                                    ])
                                ]
                            )
                        ]
                    )
                ]

    # -------------------------------------------------
    # RESET / ALL to show blank map
    # -------------------------------------------------
    if filter_value == "reset":
        return [base_map]

    # -------------------------------------------------
    # show all filtered markers
    # -------------------------------------------------
    markers = [
        dl.Marker(
            position=[row['location_lat'], row['location_long']],
            children=[
                dl.Tooltip(row['breed']),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(row['name'])
                ])
            ]
        )
        for _, row in dff.iterrows()
        if pd.notna(row['location_lat']) and pd.notna(row['location_long'])
    ]

    if not markers:
        return [base_map]

    center_lat = dff['location_lat'].mean()
    center_lon = dff['location_long'].mean()

    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[center_lat, center_lon],
            zoom=10,
            children=[dl.TileLayer()] + markers
        )
    ]
# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Number of records loaded: 10004
Columns in dataframe: ['rec_num', 'age_upon_outcome', 'animal_id', 'animal_type', 'breed', 'color', 'date_of_birth', 'datetime', 'monthyear', 'name', 'outcome_subtype', 'outcome_type', 'sex_upon_outcome', 'location_lat', 'location_long', 'age_upon_outcome_in_weeks', 'status']
Dash app running on https://marginshirt-denvervictor-3000.codio.io/proxy/8050/
